In [ ]:
import numpy
import pandas

import matplotlib.pyplot as plt
plt.style.use('mystyle.mplstyle')

import sbruceana

In [ ]:
PATH_TO_SBRUCE = "/Users/triozzi/Analysis/nuedis/checks/data/cc1e0pi/"

In [ ]:
FILE_CV = "preselection_electron/CNAF_CV_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"
FILE_OFFBEAM = ""

# using data gains from 1D 
FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron_CaloFix_2.root"

# using data gains from 2D
# FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron_CaloFix.root"

# using MC gains
# FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

width = 0.3; bins = numpy.arange(0.3, 11+width, width)

ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.plotting.CC1E0PI_CATEGORIES, bins, var)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var)

ax = sbruceana.plotting.place_cut(ax, 3.5, True)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='NuMI CV'); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_{var}.pdf", dpi=300)

In [ ]:
def sweep_chi2(
  df,
  df_data,
  var,
  bins,
  fs = numpy.arange(0.6, 1.5, 0.001)
):
  chi_sqs = []
  for f in fs:
    chi_sq, _ = sbruceana.utils.chi2_histograms(df, df_data, var, bins, f)
    chi_sqs.append(chi_sq)

  return fs, chi_sqs

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

width = 0.2; bins = numpy.arange(0.3, 11+width, width)

var = "ind1dEdx"
fs, chi_sqs = sweep_chi2(df, df_data, var, bins)
min_chi_sq = fs[numpy.argmin(chi_sqs)]
ax.plot(fs, chi_sqs, label=f'Ind-1\n{min_chi_sq:.4f}')

var = "ind2dEdx"
fs, chi_sqs = sweep_chi2(df, df_data, var, bins)
min_chi_sq = fs[numpy.argmin(chi_sqs)]
ax.plot(fs, chi_sqs, label=f'Ind-2\n{min_chi_sq:.4f}')

var = "colldEdx"
fs, chi_sqs = sweep_chi2(df, df_data, var, bins)
min_chi_sq = fs[numpy.argmin(chi_sqs)]
ax.plot(fs, chi_sqs, label=f'Coll\n{min_chi_sq:.4f}')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'data scale factor ($\\mathcal{G}=0.0133$)',
  ylabel = 'data/MC $\\chi^2$',
  xlim   = (min(fs), max(fs)),
)
leg = ax.legend(fontsize=10, title='NuMI CV'); leg.get_title().set_fontsize(11)

In [ ]:
# using MC gains
gains = numpy.array([0.01343, 0.01338, 0.01219])

# factors to obtain data/MC match
factors = numpy.array([1, 1, 0.8589])

In [ ]:
# using data gains
gains = numpy.array([0.0133333, 0.0133333, 0.0133333])

# factors to obtain data/MC match
factors = numpy.array([1.1139, 1.1333, 1.219])

In [ ]:
# Collection
0.01219 + (1 - 0.8589) / (1.219 - 0.8589) * (0.01333 - 0.01219)

In [ ]:
# Induction-1
0.01333 + (1 - 1.1139) / 315.9

In [ ]:
# Induction-2
0.01333 + (1 - 1.1333) / 315.9